# 11 · 第二个工具：查一个病的资料

## 这一章在整个项目里的位置

```
00 总览 → 01 存档旧项目 ✅ → 02 新项目骨架 ✅ → 03 自动检查 ✅ → 04 GitHub 与 CI ✅ → 05 准备考题 ✅ → 06 评测集 ✅ → 07 判分 ✅ → 08 MLflow 评测 ✅ → 09 第一个基线 ✅ → 10 第一个工具 ✅ → [11 第二个工具 ✅] → 12 第三个工具 ✅ → 13 标准接口 MCP ✅ → 14 固定流程 ✅ → 15 第一个 agent ✅ → 16 起：改进 agent …
```

> **这一章的代码在合并请求 #13 里。** 这一章不调大模型，一分钱不花。

## 背景：拿到候选以后要做什么

第 10 章的工具 `rank_diseases` 给出候选病的排名，但只有一个分数。医生拿到排名后会做什么？**一个一个翻资料**：这个病一般有哪些症状、多常见、什么遗传方式、几岁发病、哪个基因、资料出自哪里。agent 也要能做这件事。这一章的工具 `lookup_disease`（计划里的 T3）就是"翻资料"。

还记得第 10 章那道 Perrault 的题：工具把"常染色体隐性耳聋 67 型"排第 1，正确答案 Perrault 7 型排第 6。只看分数分不出谁对，翻一翻资料就清楚了（第 5 步）。

这一章没有新的分数：查资料的工具自己不排序。它的用处，要等 agent 用它的时候才看得出来。

## 这一章做的 5 件事

| # | 做什么 | 用的技术 | 解决什么问题 |
|---|---|---|---|
| 1 | 一个病，HPO 记了什么 | 同一份钉住的 HPO | 不加新数据源 |
| 2 | 把频率统一成一种说法 | 4 种写法 → 一个比例 | 能排序，也能读 |
| 3 | 回答要写给模型看 | 排序 + 上限 + 总数 + 出处 | 省 token、有重点、能引用 |
| 4 | 遮住也要生效 | 第 10 章的遮挡 | 不从另一个工具漏回答案 |
| 5 | 用起来看看 | 第 10 章那道题 | 为什么还需要"核对"工具 |

---

## 准备

运行下面这一格。它定义一个叫 `run` 的小工具，用来执行命令并显示结果。**不用看代码。**

> 运行 `uv` 的命令时，有时会先出现几行 `Building` / `Installed`：那是 uv 发现代码有变化，顺手更新了一下环境，可以忽略。

In [ ]:
import os, subprocess
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
os.environ.update(NO_COLOR="1", GH_PAGER="cat", GIT_PAGER="cat")
def run(cmd, check=True):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout + r.stderr)
    if check and r.returncode != 0:
        raise RuntimeError(f"failed: {cmd}")
print("项目目录:", ROOT)

---

## 第 1 步 · 一个病，HPO 记了什么

### 在干什么
看看 HPO 的疾病注释里，除了症状还记了什么。

### 背景
第 10 章只用了注释文件里的"症状"这一类。同一个文件里还有别的类别，每一行也都写着出处。

### 用了什么技术，为什么

| 类别 | 例子 |
|---|---|
| 症状（P） | 感音神经性耳聋，13 个病人里 13 个有 |
| 遗传方式（I） | 常染色体隐性遗传 |
| 发病和病程（C） | 儿童期发病、进行性加重 |

致病基因来自第 10 章已经在用的 `genes_to_disease.txt`。所以这个工具**不需要新的数据源**：还是那一份钉住版本的 HPO，只是读出更多内容。

### 公司里什么情况怎么做
- **先用手里已经钉好的数据**把事情做通，再考虑加新来源
- **每加一个来源**，都要走一遍第 05、10 章的流程：钉版本、核指纹、查许可、查泄漏

### 执行
在全部 OMIM 病上数一数：

In [ ]:
import statistics
from raredx.data import sources
from raredx.knowledge import hpo as hpo_knowledge

registry = sources.load_sources(ROOT / "evals/datasets.toml")
hpo = hpo_knowledge.load(sources.raw_dir(registry["hpo"], ROOT / "data"))
diseases = hpo.disease_ids

n_features = [len({x.term for x in hpo.lines(d, "P") if not hpo_knowledge.absent(x.frequency)}) for d in diseases]
print(f"OMIM 病：{len(diseases):,} 个")
print(f"每个病记了几个症状：中位数 {statistics.median(n_features):.0f} 个，最多 {max(n_features)} 个")
for aspect, name in (("I", "遗传方式"), ("C", "发病和病程")):
    known = sum(bool(hpo.lines(d, aspect)) for d in diseases)
    print(f"记了{name}的病：{known:,} 个（{known / len(diseases):.0%}）")

### 结果怎么看
- 一个病一般记了 12 个左右的症状，但最多的有两百多个
- 遗传方式几乎每个病都有（99%）；发病和病程只有一半多（57%）

"最多两百多个"这一点，第 3 步还会用到。

### 面试怎么说
> "The disease lookup reads more of the release the ranking tool already pins — inheritance, onset and course alongside phenotypes — so it adds no new source to version, verify or check for leakage."

---

## 第 2 步 · 把频率统一成一种说法

### 在干什么
"多少病人有这个症状"，文件里有四种写法。把它们统一成一个比例，才能排序。

### 背景
OMIM 病的症状注释里：

| 写法 | 例子 | 大约多少行 |
|---|---|---|
| 计数 | `3/5`：5 个病人里 3 个有 | 9.7 万（其中 3 千行是 `0/n`） |
| 不写 | （空着） | 5.2 万 |
| 频率等级 | `HP:0040282`，意思是"常见（30–79%）" | 3,500 |
| 百分比 | `25%` | 100 |

### 用了什么技术，为什么
规则写在代码里（[`src/raredx/knowledge/hpo.py`](../../src/raredx/knowledge/hpo.py) 的 `frequency`），每条规则一个测试：

1. **计数相加**：两篇论文分别是 3/4 和 1/4，合起来 4/8
2. **"0/n" 也算进去**：一篇说 2/2，另一篇查了 3 个病人都没有（0/3），合起来是 2/5，不是 100%
3. **频率等级取范围的中间**：常见（30–79%）按 55% 排序，显示时写原来的等级
4. **没写就是"不知道"**：排在最后，不编一个数
5. **所有来源都说"没有"**（只有 0/n）：不算这个病的症状（第 10 章就是这样做的）

### 公司里什么情况怎么做
- **数据统一的规则写进代码、配上测试**，不靠每次手工处理
- **"不知道"就是不知道**：不要用默认值填空、假装知道；下游（这里是模型）需要分清"没记录"和"很少见"

### 执行

In [ ]:
from raredx.knowledge.hpo import frequency

for written in (["3/4", "1/4"], ["2/2", "0/3"], ["HP:0040282"], ["25%"], [""]):
    f = frequency(written)
    shown = "不知道" if f.fraction is None else f"{f.text}（按 {f.fraction:.0%} 排序）"
    print(f"{' 和 '.join(w or '空' for w in written):<20} → {shown}")

### 结果怎么看
五行对应上面的规则：3/4 和 1/4 合成 4/8；2/2 和 0/3 合成 2/5；常见按 55% 排序；25% 原样；空着就是不知道。

### 面试怎么说
> "Frequencies come in four notations — counts, percentages, HPO frequency terms, or nothing. Counts pool across sources, including sources that found none; terms map to the middle of their range; a missing frequency stays unknown rather than defaulted. Each rule has a test."

---

## 第 3 步 · 回答要写给模型看

### 在干什么
设计工具的回答：给什么、按什么顺序、给多长。

### 背景
工具的回答会原样放进模型的上下文。代价有两个：
- **花钱**：按 token 计费，一次 agent 可能要查十几个病
- **分散注意力**：塞进一大堆不相关的内容，模型更容易漏掉关键的那一条

第 1 步看到，一个病最多记了两百多个症状。全部列出来，一次回答就是几千个 token。

### 用了什么技术，为什么

| 做法 | 为什么 |
|---|---|
| 最常见的症状排前面；一样常见的，越特异越靠前 | 模型先看到最能区分的 |
| 默认最多 20 个（`max_features` 可调），并写明一共有多少（`n_features`） | 不撑爆上下文；模型知道还有，需要时再要 |
| 给名字，不只给编号 | 模型读得懂 "Streak ovary"（条索状卵巢），读不懂 "HP:0010464"。编号也给，方便它接着调别的工具 |
| 每个症状带出处（PMID） | agent 的结论要能引用证据：哪篇论文说的 |
| 编号写错，给一个能照着改的错误 | "这个病不存在，请用 rank_diseases 返回的编号"，模型改了可以重试 |

这几条也是行业里公开的工具设计建议（例如 Anthropic 2025 年发布的工具设计指南）：返回有用的内容、用名字而不是一串编号、节省 token（截断、分页、合理的默认值）、错误信息要能指导下一步。

### 公司里什么情况怎么做
- **量最长的情况**：上线前，工具的回答最长多长要心里有数。下面把全部 8,478 个病都查一遍
- **回答格式也要评审**：改回答的格式和改提示词一样，走合并请求、重跑评测

### 执行
把每个病都查一遍，量回答有多长。查八千多次不需要一条条记录，所以先关掉 MLflow 的追踪，量完再打开：

In [ ]:
import mlflow
from raredx.config import settings
from raredx.tools.lookup_disease import LookupDisease

mlflow.set_tracking_uri(settings.mlflow_tracking_uri)
mlflow.set_experiment("playground")
lookup_disease = LookupDisease(hpo)

mlflow.tracing.disable()
default = [len(lookup_disease(d).model_dump_json()) for d in diseases]
everything = [len(lookup_disease(d, max_features=300).model_dump_json()) for d in diseases]
mlflow.tracing.enable()

for name, sizes in (("默认（最多 20 个症状）", default), ("全部列出", everything)):
    print(f"{name:<14} 中位数 {statistics.median(sizes):>6,.0f} 字符   最长 {max(sizes):>6,} 字符（约 {max(sizes) // 4:,} token）")

### 结果怎么看
- 一般的病，两种方式一样长（一千多个字符）：它本来就不到 20 个症状
- 最长的情况：全部列出约 2.7 万个字符，**约 7 千 token**；默认方式最长约 5 千多字符，大约是它的五分之一
- 英文大约 4 个字符算 1 个 token，这里是估算

### 面试怎么说
> "Tool output is written for the model: ranked by frequency then specificity, capped at a default of 20 features with the total stated, names alongside ids, a source on every feature, and errors that say how to recover. Across all 8,478 diseases the default answer is at most about 1,400 tokens, against 7,000 for the longest full dump."

---

## 第 4 步 · 遮住也要生效

### 在干什么
确认第 10 章的遮挡对这个工具也生效：考某道题时，这道题自己论文里来的资料，查不到。

### 背景
如果排名工具遮了、查资料的工具没遮，agent 一翻资料就又看到答案了：泄漏从另一个工具漏了回来。

### 用了什么技术，为什么
同一个机制（第 10 章的 `contextvars`）。这个工具读每一行注释时都看一眼遮挡设置：症状、遗传方式、发病年龄、出处，全都按行过滤。**回答里不会提到遮挡**，模型不知道有东西被遮了。有一条测试专门检查这一点。

### 公司里什么情况怎么做
- **新工具上线前过一遍泄漏检查**：它读的数据里，有没有考题的来源
- **遮挡由评测框架统一做**：新工具只要按规矩通过知识层读数据，就自动生效

### 执行
先看 Perrault 那道题遮住后查到什么；再在 dev-200 上数一数，有多少道题的正确答案，遮住之后什么都查不到：

In [ ]:
from raredx.data import cases as case_store
from raredx.evals import splits
from raredx.knowledge import masking

by_id = {c.id: c for c in case_store.read(sources.processed_dir(registry["phenopacket-store"], ROOT / "data"))}
case = by_id["PMID_39701103_Proband_F3"]
with masking.masked([case.pmid]):
    blank = lookup_disease(case.disease_id)
print(f"遮住以后查 {blank.disease_name}：症状 {blank.n_features} 个，遗传方式 {blank.inheritance}，出处 {blank.sources}，基因 {blank.genes}\n")

dev = [by_id[i] for i, _ in splits.read(ROOT / "evals/splits/dev-200.tsv")]
not_in_hpo = [c for c in dev if c.disease_id not in hpo.names]
mlflow.tracing.disable()
blank_after_mask = 0
for c in dev:
    if c.disease_id in hpo.names:
        with masking.masked([c.pmid]):
            blank_after_mask += lookup_disease(c.disease_id).n_features == 0
mlflow.tracing.enable()
print(f"dev-200 的正确答案里：HPO 根本没有这个病 {len(not_in_hpo)} 个；遮住论文后一个症状都查不到 {blank_after_mask} 个")

### 结果怎么看
- Perrault 7 型遮住以后：症状、遗传方式、出处全空，只剩名字和基因（基因来自另一份数据，不出自这篇论文）
- dev-200 里：**41 道题**（20%）的正确答案，靠 HPO 什么症状都查不到：40 个病 HPO 只从这道题自己的论文认识它，1 个病 HPO 里根本没有

这说明了两件事：
1. **按症状打分有天花板**：这 41 道题它永远答不对，最多 159/200，约 80%
2. **这些多是新报道的病**：知识库里只有一篇论文，遮住就没了。要答对它们，agent 需要**去查文献**，这是以后的一个工具

### 面试怎么说
> "The same context-variable mask covers the lookup tool line by line, and its answers never mention it. Masked, 41 of the 200 dev answers have no phenotype left in HPO — mostly newly described diseases known from a single paper — which caps phenotype-only ranking near 80% and is the case for a literature-search tool."

---

## 第 5 步 · 用起来：第 10 章那道题

### 在干什么
用这个工具翻一翻：排名第 1 的"耳聋 67 型"和正确答案 Perrault 7 型，各记了什么。

### 背景
这里用不遮的知识，就像真实门诊：那篇论文早就发表了，医生查得到。考试时会遮住（第 4 步）。

### 执行

In [ ]:
patient = set(case.hpo_ids)
print("病人的症状：", "、".join(case.hpo_labels), "\n")
for disease_id in ["OMIM:610265", case.disease_id]:  # 排名第 1 的病，正确答案
    profile = lookup_disease(disease_id)
    print(f"{profile.disease_name}（{', '.join(profile.genes)}；{'、'.join(profile.inheritance)}）共 {profile.n_features} 个症状：")
    for f in profile.features:
        mark = "  ✓ 病人有" if f.hpo_id in patient else ""
        print(f"   {f.frequency or '频率不知道':<8} {f.label}{mark}")
    print()

### 结果怎么看
- **耳聋 67 型**只记了 2 个症状，都是耳聋。病人 5 个症状里，它只解释得了耳聋
- **Perrault 7 型**记了 11 个，病人的 5 个症状全在里面（打 ✓ 的），还有卵巢、激素这些耳聋病没有的
- 按症状打分时，耳聋 67 型排第 1，是因为它的症状少而"全对"；翻资料一看就知道它解释不了闭经和卵巢功能不全

**人看一眼就明白，但要让 agent 稳定地做到，需要一个专门的工具**：把病人的症状和一个病逐条比，对上了几个、缺了哪些、有没有矛盾；"双侧感音神经性耳聋"和"感音神经性耳聋"这种上下级术语也要算对上。这就是下一章。

### 面试怎么说
> "On the worked example, the top-ranked deafness entry explains one of the patient's five findings; the correct Perrault entry explains all five. Reading the evidence separates them — which is what the agent's verification step has to do reliably."

---

## 本章小结

| 步骤 | 技术 | 公司里的作用 |
|---|---|---|
| 1 HPO 还记了什么 | 同一份钉住的 HPO | 不加新数据源 |
| 2 统一频率 | 计数相加、等级取中、不知道就写不知道 | 规则在代码里，每条有测试 |
| 3 写给模型看 | 排序 + 默认 20 个 + 总数 + 出处 + 能照着改的错误 | 省 token、有重点、能引用 |
| 4 遮挡 | 同一个 `contextvars` 遮挡，按行过滤 | 不从另一个工具漏回答案 |
| 5 用起来 | 翻资料比较两个候选 | 引出"核对"工具 |

**一句话**：第二个工具"查一个病的资料"把 HPO 里一个病的症状、频率、遗传方式、发病年龄、基因和出处，整理成模型读得动的一页；它也服从遮挡，并量出了一个新问题：20% 的正确答案是知识库只从一篇论文认识的新病。

## 面试题

1. **How do you design what a tool returns to an LLM?**
   > "For the model's context: the most useful items first, a sensible default cap with the total stated, readable names next to ids, sources for citation, and errors that say how to recover. I measure the worst case: across all 8,478 diseases the default answer is at most about 1,400 tokens."

2. **The source data writes frequencies four ways. What did you do?**
   > "Normalised in one tested function: counts pooled across sources, including zero counts; frequency terms at the middle of their range; missing stays unknown and sorts last."

3. **How do you keep a second tool from leaking the answer the first one hides?**
   > "Masking lives in the knowledge layer and is set by the evaluation harness, so every tool that reads knowledge through that layer honours it, line by line, and none of them mentions it."

4. **What did building the lookup tool tell you about the task?**
   > "That 41 of the 200 dev answers have no phenotype in HPO once the case's own paper is masked. Phenotype ranking can't exceed about 80% on this set; the rest needs literature search."

**下一章 12 · 第三个工具：核对吻合度**：病人的症状和一个病逐条比，对上的、缺的、矛盾的，各是哪些。